In [1]:
import pandas as pd
import gc
import numpy as np

In [2]:
df3 = pd.read_pickle('../data/df3.pkl')

In [ ]:
df3.info()
df3.shape # (7483231, 30)
#df3.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7483231 entries, 0 to 7483230
Data columns (total 30 columns):
 #   Column              Dtype         
---  ------              -----         
 0   BUS_SVC_NUM         float64       
 1   CRD_NUM             object        
 2   DEST_LOC_ID_NUM     float64       
 3   ENTRY_DT            datetime64[s] 
 4   ENTRY_TM            datetime64[ns]
 5   EXIT_DT             datetime64[s] 
 6   EXIT_TM             datetime64[ns]
 7   JRNY_ID_NUM         object        
 8   ORIG_LOC_ID_NUM     float64       
 9   RIDE_DISC_AMT       float64       
 10  RIDE_DIST_KM_CNT    float64       
 11  RIDE_FARE_AMT       float64       
 12  RIDE_ID_NUM         object        
 13  RIDE_MIN_CNT        float64       
 14  PATRON_CATG_ID_NUM  int64         
 15  TRNSPT_MODE_CD      int64         
 16  DEST_STATION_NAME   object        
 17  DEST_MRK_ID_NUM     float64       
 18  DEST_LATITUDE       float64       
 19  DEST_LONGITUDE      float64       
 20  DE

(7483231, 30)

### Information on the Patron Categories: 

- Student (Primary 1 to JC/Poly) --> 7 - 19 years old
- Adult --> 20 - 59 years old
- Senior Citizen --> 60 years and above

Average Walking Speeds by Age:
- Children (6-12 years): Around 3.0 to 4.5 km/h (1.9 to 2.8 mph)
- Adolescents (13-19 years): Around 4.0 to 5.0 km/h (2.5 to 3.1 mph)
- Adults (20-59 years): Around 5.0 to 6.0 km/h (3.1 to 3.7 mph)
- Older Adults (60+ years): Around 4.0 to 5.0 km/h (2.5 to 3.1 mph)

To calculate the walking speed of each group, I take the midpoint of the range. 
For Children and Adolescents, I took the midpoint of both ranges and took the average.

In [4]:
# We first filter for 3 relevant PATRON_CATG_ID_NUM.
patron_map = {1: 'Adult', 3: 'Student', 4: 'Senior Citizen'}
df3['PATRON_CATG_DESC_TXT'] = df3['PATRON_CATG_ID_NUM'].map(patron_map)

walking_speed_ms = {
    'Student': 4.125 / 3.6,      # (3.75 + 4.5) / 2 = 4.125 km/h → ~1.146 m/s
    'Adult': 5.5 / 3.6,          # ~1.528 m/s
    'Senior Citizen': 4.5 / 3.6  # 1.25 m/s
}

df3['walking_speed_ms'] = df3['PATRON_CATG_DESC_TXT'].map(walking_speed_ms)

# Rule Based Classification

#### 1. Binary Criteria
- Drop rows with missing critical data
- The journey stage has no subsequent recorded trips on the same day and is therefore treated as a journey termination. It is the last ride of the day for the commuter.
    - Implementation: df3 is already sorted by CRD_NUM and ENTRY_TM. Flag last row of each CRD_NUM
- 2 consecutive journey stages on the same bus service or train station indicates a return trip or intermediate activity. This should be classified as a new journey.
    - Implementation: Shift BUS_SVC_NUM and ORIG_LOC_ID_NUM down by 1 row to get the next stage's values, then flag rows where all 3 conditions are true simultaneously: same commuter, same bus service, and the alighting stop of stage 1 equals the boarding stop of stage 2.


In [5]:
df3 = df3.sort_values(["CRD_NUM", "ENTRY_TM"]).reset_index(drop=True)

In [6]:
df3["next_ENTRY_TM"] = df3.groupby("CRD_NUM")["ENTRY_TM"].shift(-1)
df3["next_ORIG_LOC_ID_NUM"] = df3.groupby("CRD_NUM")["ORIG_LOC_ID_NUM"].shift(-1)
df3["next_BUS_SVC_NUM"] = df3.groupby("CRD_NUM")["BUS_SVC_NUM"].shift(-1)
df3["next_TRNSPT_MODE_CD"] = df3.groupby("CRD_NUM")["TRNSPT_MODE_CD"].shift(-1)

In [7]:
df3["is_last_stage"] = df3["next_ENTRY_TM"].isna()

df3['missing_info'] = (
    df3['ENTRY_TM'].isna() |
    df3['EXIT_TM'].isna() |
    df3['ORIG_LATITUDE'].isna() |
    df3['ORIG_LONGITUDE'].isna() |
    df3['DEST_LATITUDE'].isna() |
    df3['DEST_LONGITUDE'].isna() |
    df3["ORIG_LOC_ID_NUM"].isna() |
    df3["DEST_LOC_ID_NUM"].isna()
)

In [8]:
#create next bus and next station columns by shifting
# flags consecutive rides on the same bus service number/same station
df3["same_bus_service"] = (
    df3["BUS_SVC_NUM"].notna() &
    df3["next_BUS_SVC_NUM"].notna() &
    (df3["BUS_SVC_NUM"] == df3["next_BUS_SVC_NUM"])
)
df3["same_station_consecutive"] = (
    df3["DEST_STATION_NAME"].notna() &
    df3["next_orig_station"].notna() &
    (df3["DEST_STATION_NAME"] == df3["next_orig_station"])
)

In [9]:
df3["return_or_intermediate"] = (
    df3["same_bus_service"] |
    df3["same_station_consecutive"]
)

In [10]:
df3["journey_termination_flag"] = (
    df3["is_last_stage"] |
    df3["missing_info"] |
    df3["return_or_intermediate"]
)

In [11]:
df3['journey_termination_flag'].value_counts()


journey_termination_flag
True     4493025
False    2990206
Name: count, dtype: int64

In [ ]:
#df3[df3['journey_termination_flag']].head(40)

In [12]:
# Summary counts
print("Unique commuters:", df3["CRD_NUM"].nunique())
print("Last stages:", df3["is_last_stage"].sum())

print("\nMissing info:")
print(df3["missing_info"].value_counts(dropna=False))

print("\nSame bus service:")
print(df3["same_bus_service"].value_counts(dropna=False))

print("\nSame station consecutive:")
print(df3["same_station_consecutive"].value_counts(dropna=False))

print("\nReturn / intermediate:")
print(df3["return_or_intermediate"].value_counts(dropna=False))

print("\nJourney termination flag:")
print(df3["journey_termination_flag"].value_counts(dropna=False))

Unique commuters: 2462129
Last stages: 2462129

Missing info:
missing_info
False    7414732
True       68499
Name: count, dtype: int64

Same bus service:
same_bus_service
False    6931342
True      551889
Name: count, dtype: int64

Same station consecutive:
same_station_consecutive
False    5952424
True     1530807
Name: count, dtype: int64

Return / intermediate:
return_or_intermediate
False    5486609
True     1996622
Name: count, dtype: int64

Journey termination flag:
journey_termination_flag
True     4493025
False    2990206
Name: count, dtype: int64


In [13]:
# If you want to see flag rows
'''
display(
    df3.loc[
        df3["journey_termination_flag"],
        [
            "CRD_NUM",
            "ENTRY_TM",
            "EXIT_TM",
            "ORIG_STATION_NAME",
            "DEST_STATION_NAME",
            "next_orig_station",
            "BUS_SVC_NUM",
            "next_BUS_SVC_NUM",
            "is_last_stage",
            "missing_info",
            "same_bus_service",
            "same_station_consecutive",
            "return_or_intermediate",
            "journey_termination_flag"
        ]
    ].head(20)
)
'''

'\ndisplay(\n    df3.loc[\n        df3["journey_termination_flag"],\n        [\n            "CRD_NUM",\n            "ENTRY_TM",\n            "EXIT_TM",\n            "ORIG_STATION_NAME",\n            "DEST_STATION_NAME",\n            "next_orig_station",\n            "BUS_SVC_NUM",\n            "next_BUS_SVC_NUM",\n            "is_last_stage",\n            "missing_info",\n            "same_bus_service",\n            "same_station_consecutive",\n            "return_or_intermediate",\n            "journey_termination_flag"\n        ]\n    ].head(20)\n)\n'

In [14]:
# If you want to see transfers
'''
display(
    df3.loc[
        ~df3["journey_termination_flag"],
        [
            "CRD_NUM",
            "ENTRY_TM",
            "EXIT_TM",
            "ORIG_STATION_NAME",
            "DEST_STATION_NAME",
            "next_orig_station",
            "BUS_SVC_NUM",
            "next_BUS_SVC_NUM",
            "is_last_stage",
            "missing_info",
            "same_bus_service",
            "same_station_consecutive",
            "return_or_intermediate",
            "journey_termination_flag"
        ]
    ].head(20)
)
'''

'\ndisplay(\n    df3.loc[\n        ~df3["journey_termination_flag"],\n        [\n            "CRD_NUM",\n            "ENTRY_TM",\n            "EXIT_TM",\n            "ORIG_STATION_NAME",\n            "DEST_STATION_NAME",\n            "next_orig_station",\n            "BUS_SVC_NUM",\n            "next_BUS_SVC_NUM",\n            "is_last_stage",\n            "missing_info",\n            "same_bus_service",\n            "same_station_consecutive",\n            "return_or_intermediate",\n            "journey_termination_flag"\n        ]\n    ].head(20)\n)\n'

In [13]:
# Reasons why flagged.
df3["journey_termination_reason"] = np.select(
    [
        df3["is_last_stage"],
        df3["missing_info"],
        df3["return_or_intermediate"]
    ],
    [
        "last_stage",
        "missing_info",
        "return_or_intermediate"
    ],
    default="candidate_transfer"
)

In [14]:
print(df3["journey_termination_reason"].value_counts(dropna=False))

journey_termination_reason
candidate_transfer        2990206
last_stage                2462129
return_or_intermediate    1985090
missing_info                45806
Name: count, dtype: int64


### Create "median" time gap for each bucket of walking distance (in 50m intervals). 
Group by patron cat


In [16]:
# Calculate time gap mins

# Shift next entry time within the same commuter group
df3['next_ENTRY_TM'] = df3.groupby('CRD_NUM')['ENTRY_TM'].shift(-1)

# Time gap = next stage's entry time minus current stage's exit time (in mins)
df3['time_gap_mins'] = (
    df3['next_ENTRY_TM'] - df3['EXIT_TM']
).dt.total_seconds() / 60


In [ ]:
bins = list(range(0, 1250, 50))
labels = [f"{i}-{i+50}m" for i in range(0, 1200, 50)]

# Create as object dtype so we can freely assign '0m'
df3['walk_dist_bucket'] = pd.cut(
    df3['walk_distance'],
    bins=bins,
    labels=labels,
    right=False
).astype(object)

# Override: exact 0 gets its own bucket. 0 is removed from 0-50m bucket.
df3.loc[df3['walk_distance'] == 0, 'walk_dist_bucket'] = '0m'

# Compute median time gap and count per bucket + patron category
bucket_medians = (
    df3.dropna(subset=['walk_distance', 'time_gap_mins', 'PATRON_CATG_DESC_TXT'])
    .groupby(['walk_dist_bucket', 'PATRON_CATG_DESC_TXT'], observed=True)['time_gap_mins']
    .agg(count='count', median_time_gap_mins='median')
    .reset_index()
)

print(bucket_medians.to_string())

   walk_dist_bucket PATRON_CATG_DESC_TXT    count  median_time_gap_mins
0             0-50m                Adult   104908              3.433333
1             0-50m       Senior Citizen    26249              5.383333
2             0-50m              Student    15388              3.616667
3                0m                Adult  1165544            135.433333
4                0m       Senior Citizen   321490             32.800000
5                0m              Student   168353             85.400000
6          100-150m                Adult   312735              7.316667
7          100-150m       Senior Citizen    91853             16.333333
8          100-150m              Student    69148             82.616667
9        1000-1050m                Adult     8245            226.616667
10       1000-1050m       Senior Citizen     1591            197.083333
11       1000-1050m              Student     1236            335.933333
12       1050-1100m                Adult    13432            224

In [24]:
# Map median back to df3 by bucket + patron category
df3 = df3.merge(
    bucket_medians[['walk_dist_bucket', 'PATRON_CATG_DESC_TXT', 'median_time_gap_mins']],
    on=['walk_dist_bucket', 'PATRON_CATG_DESC_TXT'],
    how='left'
)

print(df3['median_time_gap_mins'].isna().sum(), "rows with no median (null bucket or patron category)")

2872540 rows with no median (null bucket or patron category)


In [ ]:
df3.head()

In [23]:
bucket_stats_combined = (
    df3.dropna(subset=['walk_distance', 'time_gap_mins'])
    .groupby('walk_dist_bucket', observed=True)['time_gap_mins']
    .agg(count='count', median_time_gap_mins='median')
    .reset_index()
)

print(bucket_stats_combined.to_string())

   walk_dist_bucket    count  median_time_gap_mins
0             0-50m   151419              3.733333
1                0m  1707846             99.300000
2          100-150m   489484              9.866667
3        1000-1050m    11365            230.883333
4        1050-1100m    16813            222.033333
5        1100-1150m    11066            290.300000
6        1150-1200m     9279            290.600000
7          150-200m   542427              6.300000
8          200-250m   343857              7.450000
9          250-300m   242816             11.183333
10         300-350m   209477             24.116667
11         350-400m   143222             43.633333
12         400-450m   113160             53.216667
13         450-500m    75068            122.125000
14          50-100m   367400              5.700000
15         500-550m    64549            133.700000
16         550-600m    46711            132.300000
17         600-650m    43539            208.016667
18         650-700m    30905   

#### 2. Temporal criteria: 
- The binary criteria gives us pairs of consecutive stages that could be transfers. 
- Temporal criteria finds the time gap between alighting from stage 1 and boarding stage 2
- Implementation: 
    - gap = ENTRY_TM of stage 2 - EXIT_TM of stage 1. 
    - Shift ENTRY_TM and TRNSPT_MODE_CD down by 1 row to get the next stage's values. Compute the time gap between current EXIT_TM and next ENTRY_TM. Assign threshold (15 or 45 mins) based on transfer type, then flag if gap exceeds it.
    - Flag if gap > allowance.
    - Allowance = Walking speed * walking distance



* Note this is how TRNSPRT_MODE_CD is mapped.
1    BUS
2    RTS
3    BUS-RTS (interchange)

In [26]:
# Flag: time_gap > median of its bucket+patron category → new journey
# Null median (no bucket or patron category match) → fillna(True) marks as new journey
df3['exceeds_time_allowance'] = (
    df3['time_gap_mins'] > df3['median_time_gap_mins']
).fillna(True)

print(df3['exceeds_time_allowance'].value_counts())
print(f"\nTotal temporal new journey flags: {df3['exceeds_time_allowance'].sum():,}")
print(f"Null median (no bucket/patron match): {df3['median_time_gap_mins'].isna().sum():,}")
print(f"Total rows: {len(df3):,}")

exceeds_time_allowance
False    5180421
True     2302810
Name: count, dtype: int64

Total temporal new journey flags: 2,302,810
Null median (no bucket/patron match): 2,872,540
Total rows: 7,483,231


In [27]:
df3['temporal_flag_reason'] = np.select(
    [
        df3['time_gap_mins'].isna(),
        df3['median_time_gap_mins'].isna(),
        df3['time_gap_mins'] > df3['median_time_gap_mins'],
    ],
    [
        'null_time_gap',
        'null_median_no_bucket_or_patron',
        'exceeds_bucket_median',
    ],
    default='within_median'
)

df3['exceeds_time_allowance'] = df3['temporal_flag_reason'] != 'within_median'

print(df3['temporal_flag_reason'].value_counts(dropna=False))

temporal_flag_reason
null_time_gap                      2507790
within_median                      2304296
exceeds_bucket_median              2302810
null_median_no_bucket_or_patron     368335
Name: count, dtype: int64


In [ ]:
# # PREVIOUS WAY OF TEMPORAL

# # allowance (seconds) = walking distance (m) / walking speed (m/s), then convert to mins
# # Rows with no walking distance → allowance is NaN  
# df3['transfer_allowance_mins'] = (
#     df3['walk_distance'] / df3['walking_speed_ms']
# ) / 60

# # True = gap exceeds allowance → new journey (not a transfer)
# # If walk_distance is null → allowance is NaN → comparison returns NaN → fillna(True) marks as new journey
# df3['exceeds_time_allowance'] = (
#     (df3['time_gap_mins'] > df3['transfer_allowance_mins'])
# ).fillna(True)

# print(df3['exceeds_time_allowance'].value_counts())
# print(f"\nTotal temporal new journey flags: {df3['exceeds_time_allowance'].sum():,}")

# print(f"Null walking distance: {df3['walk_distance'].isna().sum():,}")
# print(f"Total rows: {len(df3):,}")
# print(f"Null %: {df3['walk_distance'].isna().mean() * 100:.2f}%")

exceeds_time_allowance
True     2327847
False    1518937
Name: count, dtype: int64

Total temporal new journey flags: 2,327,847
Null walking distance: 1,239,514
Total rows: 3,846,784
Null %: 32.22%


#### 3. Spatial Criteria (Reasonable walking dist vs Actual walking dist)
- The spatial rule identifies whether the transition between consecutive journey stages is likely a transfer or the start of a new journey based on walking distance.
- It flags if the distance between the alighting location of the first journey stage and the boarding location of the next journey stage exceeds a reasonable transfer walking distance.
- Implementation: 
    - Reasonable transfer wwalking distance is the column, walking distance
    - Actual walking distance is time gap between tap out and the next tap in * walking speed
    - To account for time where commuter might be waiting for the next ride instead of walking, we include a 20% buffer.

In [31]:
# actual walking distance (m) = time gap (seconds) × walking speed (m/s)
# time_gap_mins already exists from temporal step, convert back to seconds
df3['actual_walking_dist_m'] = (
    df3['time_gap_mins'] * 60
) * df3['walking_speed_ms']

# Allowance = walk_distance (m) with 20% buffer
# If walk_distance is null → fillna(True) flags as new journey
df3['exceeds_walking_dist'] = (
    df3['actual_walking_dist_m'] > df3['walk_distance'] * 1.2
).fillna(True)

print(df3['exceeds_walking_dist'].value_counts())
print(f"\nTotal spatial new journey flags: {df3['exceeds_walking_dist'].sum():,}")

exceeds_walking_dist
True     4366060
False    3117171
Name: count, dtype: int64

Total spatial new journey flags: 4,366,060


## Hard geographic threshold (consider)

In [32]:
# Flag if the walking distance between stops is too far for a reasonable transfer
SPATIAL_THRESHOLD_M = 1200  # adjust as needed

df3['exceeds_walking_dist_threshold'] = (
    df3['walk_distance'] > SPATIAL_THRESHOLD_M
).fillna(True)  # null walk_distance → can't verify → flag as new journey

print(df3['exceeds_walking_dist_threshold'].value_counts())
print(f"\nTotal spatial new journey flags: {df3['exceeds_walking_dist_threshold'].sum():,}")

exceeds_walking_dist_threshold
False    7252565
True      230666
Name: count, dtype: int64

Total spatial new journey flags: 230,666


In [33]:
df3['spatial_flag_reason'] = np.select(
    [
        df3['walk_distance'].isna(),
        df3['walk_distance'] > SPATIAL_THRESHOLD_M,
    ],
    [
        'null_walk_distance',
        'exceeds_walking_dist_threshold',
    ],
    default='within_distance_threshold'
)

df3['exceeds_walking_dist_threshold'] = df3['spatial_flag_reason'] != 'within_distance_threshold'

print(df3['spatial_flag_reason'].value_counts(dropna=False))

spatial_flag_reason
within_distance_threshold         4759485
null_walk_distance                2493080
exceeds_walking_dist_threshold     230666
Name: count, dtype: int64


### Combining all 3 criteria

In [37]:
# Old spatial
df3['combined_flag1'] = (
    df3['journey_termination_flag'] |
    df3['exceeds_time_allowance'] |
    df3['exceeds_walking_dist']
)

df3['combined_flag1_reason'] = np.select(
    [
        df3['journey_termination_flag'],
        df3['exceeds_time_allowance'],
        df3['exceeds_walking_dist'],
    ],
    [
        'binary',
        'temporal',
        'spatial',
    ],
    default='transfer'
)

print(df3['combined_flag1'].value_counts())
print()
print(df3['combined_flag1_reason'].value_counts(dropna=False))

combined_flag1
True     7053970
False     429261
Name: count, dtype: int64

combined_flag1_reason
binary      4493025
temporal    1425690
spatial     1135255
transfer     429261
Name: count, dtype: int64


In [36]:
# New spatial
df3['combined_flag2'] = (
    df3['journey_termination_flag'] |
    df3['exceeds_time_allowance'] |
    df3['exceeds_walking_dist_threshold']
)

df3['combined_flag2_reason'] = np.select(
    [
        df3['journey_termination_flag'],
        df3['exceeds_time_allowance'],
        df3['exceeds_walking_dist_threshold'],
    ],
    [
        'binary',
        'temporal',
        'spatial',
    ],
    default='transfer'
)

print(df3['combined_flag2'].value_counts())
print()
print(df3['combined_flag2_reason'].value_counts(dropna=False))

combined_flag2
True     5918715
False    1564516
Name: count, dtype: int64

combined_flag2_reason
binary      4493025
transfer    1564516
temporal    1425690
Name: count, dtype: int64


In [53]:
# Binary + Temporal only
df3['combined_flag3'] = (
    df3['journey_termination_flag'] |
    df3['exceeds_time_allowance']
)

df3['combined_flag3_reason'] = np.select(
    [
        df3['journey_termination_flag'],
        df3['exceeds_time_allowance'],
    ],
    [
        'binary',
        'temporal',
    ],
    default='transfer'
)

print(df3['combined_flag3'].value_counts())
print()
print(df3['combined_flag3_reason'].value_counts(dropna=False))

combined_flag3
True     5918715
False    1564516
Name: count, dtype: int64

combined_flag3_reason
binary      4493025
transfer    1564516
temporal    1425690
Name: count, dtype: int64


## Internal Validity Check with pt_jrny

In [38]:
df5 = pd.read_pickle('../data/df5.pkl')
df5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5336605 entries, 0 to 5336604
Data columns (total 23 columns):
 #   Column              Dtype         
---  ------              -----         
 0   CRD_NUM             object        
 1   JRNY_DEST_ID_NUM    float64       
 2   JRNY_START_DT       datetime64[ns]
 3   JRNY_START_TM       datetime64[ns]
 4   JRNY_END_DT         datetime64[ns]
 5   JRNY_END_TM         datetime64[ns]
 6   JRNY_ORIG_ID_NUM    float64       
 7   JRNY_DIST_KM_CNT    float64       
 8   JRNY_FARE_AMT       float64       
 9   JRNY_ID_NUM         object        
 10  JRNY_TM_MIN_CNT     float64       
 11  PATRON_CATG_ID_NUM  float64       
 12  TRNSPT_MODE_CD      int64         
 13  DEST_STATION_NAME   object        
 14  DEST_MRK_ID_NUM     float64       
 15  DEST_LATITUDE       float64       
 16  DEST_LONGITUDE      float64       
 17  DEST_Travel_Type    object        
 18  ORIG_STATION_NAME   object        
 19  ORIG_MRK_ID_NUM     float64       
 20  OR

In [39]:
# same filtering as df3
df5 = df5[df5['PATRON_CATG_ID_NUM'].isin([1, 3, 4])].reset_index(drop=True)

In [40]:
# Merge on CRD_NUM + JRNY_ID_NUM to attach journey boundaries to each ride
df_val = df3.merge(
    df5[['CRD_NUM', 'JRNY_ID_NUM', 'JRNY_START_TM', 'JRNY_END_TM']],
    on=['CRD_NUM', 'JRNY_ID_NUM'],
    how='inner'
)

# Keep only rides that fall within their journey's time window
df_val = df_val[
    (df_val['ENTRY_TM'] >= df_val['JRNY_START_TM']) &
    (df_val['EXIT_TM'] <= df_val['JRNY_END_TM'])
].reset_index(drop=True)

print(f"Rows after merge + time filter: {len(df_val):,}")

Rows after merge + time filter: 7,200,557


In [41]:
df_val = df_val.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)
# Shift JRNY_ID_NUM within each commuter to get the next ride's journey ID
df_val['next_JRNY_ID_NUM'] = df_val.groupby('CRD_NUM')['JRNY_ID_NUM'].shift(-1)

# true_transfer = True → same journey (actual transfer)
# true_transfer = False → different journey (actual new journey)
df_val['true_transfer'] = (df_val['JRNY_ID_NUM'] == df_val['next_JRNY_ID_NUM'])

# Drop last stage of each commuter — no next ride to compare against
df_val = df_val[df_val['next_JRNY_ID_NUM'].notna()].reset_index(drop=True)

print(df_val['true_transfer'].value_counts())

true_transfer
False    2607966
True     2222713
Name: count, dtype: int64


In [42]:
# helper function
def print_metrics(df, pred_flag_col, label):
    """
    pred_flag_col: True = classifier says NEW JOURNEY, False = classifier says TRANSFER
    true_transfer: True = actual transfer, False = actual new journey
    """
    actual_transfer = df['true_transfer']
    pred_transfer = ~df[pred_flag_col]      # flip: flag=True means new journey, so ~flag = predicted transfer

    tp = (actual_transfer & pred_transfer).sum()     # actual transfer, predicted transfer
    tn = (~actual_transfer & ~pred_transfer).sum()   # actual new journey, predicted new journey
    fp = (~actual_transfer & pred_transfer).sum()    # actual new journey, predicted transfer (merge error)
    fn = (actual_transfer & ~pred_transfer).sum()    # actual transfer, predicted new journey (split error)

    total_actual_transfers = actual_transfer.sum()

    print(f"\n=== {label} ===")
    print(f"TP: {tp:,}  TN: {tn:,}  FP: {fp:,}  FN: {fn:,}")
    print(f"Split rate  (FN / actual transfers): {fn / total_actual_transfers:.4f}")
    print(f"Merge rate  (FP / actual transfers): {fp / total_actual_transfers:.4f}")
    print(f"Sensitivity (TP / (TP+FN)):          {tp / (tp + fn):.4f}")
    print(f"Specificity (TN / (TN+FP)):          {tn / (tn + fp):.4f}")
    print(f"Accuracy    ((TP+TN) / total):       {(tp + tn) / len(df):.4f}")

    print(pd.crosstab(
        actual_transfer,
        pred_transfer,
        rownames=['Actual transfer'],
        colnames=[f'Predicted transfer ({label})']
    ))

In [44]:
print_metrics(df_val, 'journey_termination_flag', 'Binary Criteria')


=== Binary Criteria ===
TP: 1,848,596  TN: 1,548,438  FP: 1,059,528  FN: 374,117
Split rate  (FN / actual transfers): 0.1683
Merge rate  (FP / actual transfers): 0.4767
Sensitivity (TP / (TP+FN)):          0.8317
Specificity (TN / (TN+FP)):          0.5937
Accuracy    ((TP+TN) / total):       0.7032
Predicted transfer (Binary Criteria)    False    True 
Actual transfer                                       
False                                 1548438  1059528
True                                   374117  1848596


In [45]:
print_metrics(df_val, 'exceeds_time_allowance', 'Temporal Criteria')


=== Temporal Criteria ===
TP: 1,770,954  TN: 2,062,579  FP: 545,387  FN: 451,759
Split rate  (FN / actual transfers): 0.2032
Merge rate  (FP / actual transfers): 0.2454
Sensitivity (TP / (TP+FN)):          0.7968
Specificity (TN / (TN+FP)):          0.7909
Accuracy    ((TP+TN) / total):       0.7936
Predicted transfer (Temporal Criteria)    False    True 
Actual transfer                                         
False                                   2062579   545387
True                                     451759  1770954


In [ ]:
# first spatial
print_metrics(df_val, 'exceeds_walking_dist', 'Spatial Criteria')


=== Spatial Criteria ===
TP: 430,023  TN: 2,578,238  FP: 29,728  FN: 1,792,690
Split rate  (FN / actual transfers): 0.8065
Merge rate  (FP / actual transfers): 0.0134
Sensitivity (TP / (TP+FN)):          0.1935
Specificity (TN / (TN+FP)):          0.9886
Accuracy    ((TP+TN) / total):       0.6227
Predicted transfer (Spatial Criteria)    False   True 
Actual transfer                                       
False                                  2578238   29728
True                                   1792690  430023


In [48]:
# second spatial
print_metrics(df_val, 'exceeds_walking_dist_threshold', 'Spatial Criteria')


=== Spatial Criteria ===
TP: 2,210,338  TN: 201,841  FP: 2,406,125  FN: 12,375
Split rate  (FN / actual transfers): 0.0056
Merge rate  (FP / actual transfers): 1.0825
Sensitivity (TP / (TP+FN)):          0.9944
Specificity (TN / (TN+FP)):          0.0774
Accuracy    ((TP+TN) / total):       0.4993
Predicted transfer (Spatial Criteria)   False    True 
Actual transfer                                       
False                                  201841  2406125
True                                    12375  2210338


In [ ]:
print_metrics(df_val, 'combined_flag1', 'Combined Criteria')


=== Combined Criteria ===
TP: 418,887  TN: 2,595,591  FP: 12,375  FN: 1,803,826
Split rate  (FN / actual transfers): 0.8115
Merge rate  (FP / actual transfers): 0.0056
Sensitivity (TP / (TP+FN)):          0.1885
Specificity (TN / (TN+FP)):          0.9953
Accuracy    ((TP+TN) / total):       0.6240
Predicted transfer (Combined Criteria)    False   True 
Actual transfer                                        
False                                   2595591   12375
True                                    1803826  418887


In [50]:
print_metrics(df_val, 'combined_flag2', 'Combined Criteria')


=== Combined Criteria ===
TP: 1,403,441  TN: 2,437,352  FP: 170,614  FN: 819,272
Split rate  (FN / actual transfers): 0.3686
Merge rate  (FP / actual transfers): 0.0768
Sensitivity (TP / (TP+FN)):          0.6314
Specificity (TN / (TN+FP)):          0.9346
Accuracy    ((TP+TN) / total):       0.7951
Predicted transfer (Combined Criteria)    False    True 
Actual transfer                                         
False                                   2437352   170614
True                                     819272  1403441


In [55]:
print_metrics(df_val, 'combined_flag3', 'Combined Criteria')

KeyError: 'combined_flag3'

In [58]:
# How much of df_val has null walk_distance?
print("Null walk_distance in df_val:")
print(df_val['walk_distance'].isna().sum(), '/', len(df_val))
print(f"{df_val['walk_distance'].isna().mean()*100:.1f}%")

# Of the true transfers, how many have null walk_distance?
true_transfers = df_val[df_val['true_transfer'] == True]
print("\nNull walk_distance among actual transfers:")
print(true_transfers['walk_distance'].isna().sum(), '/', len(true_transfers))
print(f"{true_transfers['walk_distance'].isna().mean()*100:.1f}%")

Null walk_distance in df_val:
6091 / 2512971
0.2%

Null walk_distance among actual transfers:
2073 / 1188828
0.2%


#ignore below

#### 3. Spatial Criteria (Haversine):
- The spatial rule identifies whether the transition between consecutive journey stages is likely a transfer or the start of a new journey based on walking distance.
- Implementation: 
    - Use original clean df to access logtitude and latitude columns
    - Sort the dataset by card number (CRD_NUM) and timestamp (ENTRY_TM) to get consecutive stages for each rider.
    - For each ride, we look at the next stage’s boarding location (latitude and longitude). Rows without a next stage are ignored because there is nothing to compare. The person finished all their trips for the day. 
    - We create a boolean mask to select only rows that have a next stage (NEXT_ENTRY_LAT is not NaN). This helps us to ignore observations without next stages
    - Use Haversine formula to compute the shortest straight-line distance between DEST_LATITUDE & DEST_LONGTITUDE (alighting stop) and the NEXT_ENTRY_LONG & NEXT_ENTRY_LAT (the next entry point). Vectorisation is used because row-by-row computation was too slow. 
    - Allow a maximum transfer distance of 500 metres (we can change this)
    - If the distance exceeds a maximum allowed transfer distance (0.5 km), the row is flagged as spatial_new_journey = True. This indicates that the next ride is likely a new journey.

In [16]:
# Shift boarding coordinates to get "next stage" per rider
df3['NEXT_ENTRY_LAT'] = df3.groupby('CRD_NUM')['ORIG_LATITUDE'].shift(-1)
df3['NEXT_ENTRY_LON'] = df3.groupby('CRD_NUM')['ORIG_LONGITUDE'].shift(-1)

In [17]:
import numpy as np
#use vectorized function so it runs much faster than row by row
def haversine_vectorized(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    
    r = 6371  # Earth radius in km
    return c * r

In [18]:
#compute distance to next stage if it exists
mask = df3['NEXT_ENTRY_LAT'].notna()  # skip last stage (ie. if there is no next tap in) 
df3.loc[mask, 'distance_to_next_km'] = haversine_vectorized(
    df3.loc[mask, 'DEST_LATITUDE'].values,
    df3.loc[mask, 'DEST_LONGITUDE'].values,
    df3.loc[mask, 'NEXT_ENTRY_LAT'].values,
    df3.loc[mask, 'NEXT_ENTRY_LON'].values
)

In [19]:
#flag spatial transfers
# set max reasonable walking transfer distance = 0.5 km 
# we can change this threshold
max_transfer_distance_km = 0.5

#initialise as False. Only mark as True if distance to next stage exceeds threshold
df3['spatial_new_journey'] = False
df3.loc[mask, 'spatial_new_journey'] = df3.loc[mask, 'distance_to_next_km'] > max_transfer_distance_km

In [20]:
df3['spatial_new_journey'].value_counts()

spatial_new_journey
False    3657236
True      189548
Name: count, dtype: int64

# Internal Validity Check
- Drop rows with missing critical data
- Combine `pt_ride` and `pt_jrny` and only keep those where the ride entry time >= journey start time and ride exit time <= journey end time
- Generate confusion matrix calculate metrics (split rate, merge rate, sensitivity, specificity, accuracy)

In [ ]:
df5 = pd.read_pickle('../data/df5.pkl')

In [ ]:
df5.info()

In [ ]:
# Drop rows with missing critical data
critical_cols = [
    'JRNY_START_TM', 'JRNY_END_TM',
    'JRNY_ORIG_ID_NUM', 'JRNY_DEST_ID_NUM',
    'ORIG_LATITUDE', 'ORIG_LONGITUDE',
    'DEST_LATITUDE', 'DEST_LONGITUDE'
]

before = len(df5)
df5 = df5.dropna(subset=critical_cols).reset_index(drop=True)
after = len(df5)

print(f"Rows dropped: {before - after:,}")
print(f"Rows remaining: {after:,}")

In [ ]:
df4 = df5[df5['JRNY_START_DT'] == '2025-02-11']

#### 1. Binary Criteria

In [ ]:
# Combine pt_ride and pt_jrny and only keep those where the ride entry time = 
df6 = df2.merge(df4, on=['CRD_NUM', 'JRNY_ID_NUM'], suffixes=('', '_JRNY'))
df6 = df6[(df6['ENTRY_DT'] >= df6['JRNY_START_DT']) & (df6['EXIT_DT'] <= df6['JRNY_END_DT'])]

In [ ]:
# Shift needed columns
df6['next_JRNY_ID_NUM'] = df6['JRNY_ID_NUM'].shift(-1)

# Flag true single journeys
df6['true_same_journey'] = (df6['JRNY_ID_NUM'] == df6['next_JRNY_ID_NUM'])

print(df6['true_same_journey'].value_counts())

In [ ]:
# Confusion matrix
pd.crosstab(
    df6['true_same_journey'],
    df6['same_service_return'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
total_true = (df6['true_same_journey'] == True).sum()

# Split Error Rate (FN)
split_error = ((df6['true_same_journey'] == True) & (df6['same_service_return'] == False)).sum()
split_rate = split_error / total_true
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df6['true_same_journey'] == False) & (df6['same_service_return'] == True)).sum()
merge_rate = merge_error / total_true
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df6['true_same_journey'] == True) & (df6['same_service_return'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df6['true_same_journey'] == False) & (df6['same_service_return'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true + merge_error + tn)
print("Accuracy:", accuracy)

#### 2. Temporal Criteria

In [ ]:
df6['true_new_journey'] = (df6['JRNY_ID_NUM'] != df6['next_JRNY_ID_NUM'])

In [ ]:
# Confusion matrix
pd.crosstab(
    df6['true_new_journey'],
    df6['temporal_new_journey'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
# Split Error Rate (FN)
split_error = ((df6['true_new_journey'] == True) & (df6['temporal_new_journey'] == False)).sum()
split_rate = split_error / total_true
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df6['true_new_journey'] == False) & (df6['temporal_new_journey'] == True)).sum()
merge_rate = merge_error / total_true
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df6['true_new_journey'] == True) & (df6['temporal_new_journey'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df6['true_new_journey'] == False) & (df6['temporal_new_journey'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true + merge_error + tn)
print("Accuracy:", accuracy)

#### 3. Spatial Criteria

In [ ]:
# Combine pt_ride and pt_jrny and only keep those where the ride entry time = 
df7 = df3.merge(df5, on=['CRD_NUM', 'JRNY_ID_NUM'], suffixes=('', '_JRNY'))
df7 = df7[(df7['ENTRY_DT'] >= df7['JRNY_START_DT']) & (df7['EXIT_DT'] <= df7['JRNY_END_DT'])]

In [ ]:
# Shift needed columns
df7['next_JRNY_ID_NUM'] = df7['JRNY_ID_NUM'].shift(-1)

# Flag true single journeys
df7['true_same_journey'] = (df7['JRNY_ID_NUM'] == df7['next_JRNY_ID_NUM'])

print(df7['true_same_journey'].value_counts())

In [ ]:
# Confusion matrix
pd.crosstab(
    df7['true_new_journey'],
    df7['spatial_new_journey'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
total_true2 = (df7['true_same_journey'] == True).sum()

# Split Error Rate (FN)
split_error = ((df7['true_new_journey'] == True) & (df7['spatial_new_journey'] == False)).sum()
split_rate = split_error / total_true2
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df7['true_new_journey'] == False) & (df7['spatial_new_journey'] == True)).sum()
merge_rate = merge_error / total_true2
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df7['true_new_journey'] == True) & (df7['spatial_new_journey'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df7['true_new_journey'] == False) & (df7['spatial_new_journey'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true2 + merge_error + tn)
print("Accuracy:", accuracy)